# Random Forest

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
import pandas as pd
import numpy as np


def train_random_forest(X, y): 
    depths = [100, 500, 1000, 2000]
    min_splits = [2, 5, 10, 15, 20]
    criterions  = ['gini', 'entropy']

    filas = []
    for criterion in criterions: 
        for min_split in min_splits: 
            for depth in depths: 
                
                accs_train, accs_val, f1s, recalls = [], [], [], []       
                kf = KFold(n_splits=5)
                
                for train_index, val_index in kf.split(X):
                    xtrain, xval = X[train_index], X[val_index]
                    ytrain, yval = y[train_index], y[val_index]
                    
                    model = RandomForestClassifier(
                        criterion=criterion,
                        min_samples_split=min_split,
                        max_depth=depth, 
                        random_state=0) 

                    model.fit(xtrain, ytrain)

                    predictions_train = model.predict(xtrain)
                    predictions_val = model.predict(xval)
                    
                    accs_train.append(accuracy_score(ytrain, predictions_train))
                    accs_val.append(accuracy_score(yval, predictions_val))
                    f1s.append(f1_score(yval, predictions_val))
                    recalls.append(recall_score(yval, predictions_val))
                    
                filas.append({
                    'maxima profundidad': depth,
                    'criterio de split': criterion, 
                    'minimo de muestras para split': min_split,
                    'media acc train': np.mean(accs_train),
                    'media acc val': np.mean(accs_val),
                    'IC acc val': np.std(accs_val),
                    'media f1': np.mean(f1s),
                    'IC f1': np.std(f1s),
                    'media recall': np.mean(recalls),
                    'IC recal': np.std(recalls)
                })

    results = pd.DataFrame(filas)
    return results.sort_values(by='media f1', ascending=False)

In [ ]:
# Para datos sin filtrar

from libs.data_configuration import get_data_partitioned

X_train, X_test, y_train, y_test = get_data_partitioned() # por defecto 20% para validacion

train_random_forest(X_train, y_train).head(10)

,maxima profundidad,criterio de split,minimo de muestras para split,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recal
23,2000,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
22,1000,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
21,500,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
20,100,entropy,2,1.000000,0.981875,0.002054,0.982078,0.001720,0.981922,0.000971
0,100,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
1,500,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
2,1000,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
3,2000,gini,2,1.000000,0.981250,0.001425,0.981471,0.001068,0.982657,0.001841
27,2000,entropy,5,0.998656,0.980250,0.000848,0.980455,0.000949,0.981666,0.002633
26,1000,entropy,5,0.998656,0.980250,0.000848,0.980455,0.000949,0.981666,0.002633


In [4]:
# para datos filtrados

from libs.data_configuration import get_filtered_data_partitioned

X_train, X_test, y_train, y_test = get_filtered_data_partitioned() # 20% de los datos para test por defecto 

train_random_forest(X_train, y_train).head(10) # mostrar los 10 mejores

,maxima profundidad,criterio de split,minimo de muestras para split,media acc train,media acc val,IC acc val,media f1,IC f1,media recall,IC recal
0,100,gini,2,1.000000,0.980625,0.002652,0.980670,0.002268,0.980076,0.004767
1,500,gini,2,1.000000,0.980625,0.002652,0.980670,0.002268,0.980076,0.004767
2,1000,gini,2,1.000000,0.980625,0.002652,0.980670,0.002268,0.980076,0.004767
3,2000,gini,2,1.000000,0.980625,0.002652,0.980670,0.002268,0.980076,0.004767
4,100,gini,5,0.997687,0.980375,0.003026,0.980449,0.002579,0.980361,0.004634
5,500,gini,5,0.997687,0.980375,0.003026,0.980449,0.002579,0.980361,0.004634
6,1000,gini,5,0.997687,0.980375,0.003026,0.980449,0.002579,0.980361,0.004634
7,2000,gini,5,0.997687,0.980375,0.003026,0.980449,0.002579,0.980361,0.004634
20,100,entropy,2,1.000000,0.980125,0.002663,0.980207,0.002180,0.980835,0.003674
21,500,entropy,2,1.000000,0.980125,0.002663,0.980207,0.002180,0.980835,0.003674


Los dos mejores modelos son los que tienen los siguientes parametros (maxima profundidad, criterio de split, minimo de muestras para split), (100, gini, 2) y (500, gini 2) sucesivamente 

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from libs.linear_reduction import train_models

xtrain, xtest, ytrain, ytest = get_filtered_data_partitioned() 

print(f"---Iniciamos con una cantidad de {xtrain.shape[1]} features---")

models = [
    RandomForestClassifier(criterion='gini', max_depth=100, min_samples_split=2),
    RandomForestClassifier(criterion='gini', max_depth=500, min_samples_split=2)
]

scaler = StandardScaler() 
xtrain, xtest = scaler.fit_transform(xtrain), scaler.transform(xtest)

pca = PCA(n_components=.95) # conservar un 95% de variabilidad de los datos 
xtrain, xtest = pca.fit_transform(xtrain), pca.transform(xtest)

print(f"---Terminamos con una cantidad de {xtrain.shape[1]} features---")

train_models(xtrain, xtest, ytrain, ytest, models, ['criterion', 'max_depth', 'min_samples_split'])

---Iniciamos con una cantidad de 41 features---
---Terminamos con una cantidad de 32 features---


,criterion,max_depth,min_samples_split,acc train,acc test,f1_score,recall
0,gini,100,2,1.0,0.9485,0.949976,0.954146
1,gini,500,2,1.0,0.9525,0.953951,0.960000
